In [ ]:
!pip install -q rank_bm25

In [ ]:
!git clone https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git

Cloning into 'multilingual-rag-research'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 29 (delta 7), reused 15 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 296.12 KiB | 24.68 MiB/s, done.
Resolving deltas: 100% (7/7), done.


In [ ]:
%cd multilingual-rag-research

/content/multilingual-rag-research


In [ ]:
!git switch nehna

branch 'nehna' set up to track 'origin/nehna'.
Switched to a new branch 'nehna'


In [ ]:
%cd /content/multilingual-rag-research

/content/multilingual-rag-research


In [ ]:
!git branch

  main
* nehna


In [ ]:
!git commit -m "Add BM25 retrieval notebook"

[nehna 399d5ab] Add BM25 retrieval notebook
 1 file changed, 152 insertions(+)
 create mode 100644 notebooks/02_bm25.ipynb


In [ ]:
  git config --global user.email "you@g.com"
  git config --global user.name "nehnamehranmk638-dev"


In [40]:
import json
from rank_bm25 import BM25Okapi

with open("data/corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

with open("data/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

print(len(corpus), len(questions))

1000 100


In [41]:
import re

def simple_tokenize(text):
    text = text.lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

# try it
print(simple_tokenize("The Eiffel Tower was completed in 1889."))

['the', 'eiffel', 'tower', 'was', 'completed', 'in', '1889']


In [42]:
tokenized_corpus = [simple_tokenize(doc["text"]) for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

In [43]:
def bm25_search(query, k=5):
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)          # one score per passage
    ranked_ids = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked_ids[:k]

# try it on your first question
q = questions[0]
print("Question:", q["question"])
print("Gold passage id:", q["gold_passage_id"])
print("Top 5 retrieved:", bm25_search(q["question"], k=5))

Question: What type of surnames is their a strong presence of?
Gold passage id: 956
Top 5 retrieved: [956, 143, 96, 177, 308]


In [44]:
all_results = {}
for q in questions:
    all_results[q["question_id"]] = bm25_search(q["question"], k=10)

In [45]:
def recall_at_k(questions, all_results, k):
    hits = 0

    for q in questions:
        retrieved = all_results[q["question_id"]][:k]

        if q["gold_passage_id"] in retrieved:
            hits += 1

    return hits / len(questions)

In [46]:
for k in [1, 3, 5, 10]:
    print(
        f"Recall@{k}:",
        recall_at_k(questions, all_results, k)
    )

Recall@1: 0.79
Recall@3: 0.88
Recall@5: 0.91
Recall@10: 0.95


In [47]:
fake_questions = [
    {
        "question_id": "x",
        "gold_passage_id": 2
    }
]

fake_results = {
    "x": [5, 2, 9]
}

print(recall_at_k(fake_questions, fake_results, k=1))
print(recall_at_k(fake_questions, fake_results, k=2))

0.0
1.0


In [48]:
def mrr(questions, all_results):
    total = 0.0
    for q in questions:
        retrieved = all_results[q["question_id"]]
        if q["gold_passage_id"] in retrieved:
            rank = retrieved.index(q["gold_passage_id"]) + 1   # 1-based rank
            total += 1.0 / rank
        # else: contributes 0
    return total / len(questions)

print("MRR:", mrr(questions, all_results))

MRR: 0.8459682539682539


In [49]:
STOPWORDS = {"the", "is", "of", "a", "an", "in", "on", "at", "to", "and", "was", "were", "for"}

def tokenize_no_stopwords(text):
    tokens = simple_tokenize(text)
    return [t for t in tokens if t not in STOPWORDS]

In [50]:
tokenized_corpus = [
    tokenize_no_stopwords(doc["text"])
    for doc in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

In [51]:
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.3)

all_results_b03 = {}

for q in questions:
    all_results_b03[q["question_id"]] = bm25_search(q["question"], k=10)

for k in [1, 3, 5, 10]:
    print(f"Recall@{k}:", recall_at_k(questions, all_results_b03, k))

print("MRR:", mrr(questions, all_results_b03))

Recall@1: 0.72
Recall@3: 0.88
Recall@5: 0.94
Recall@10: 0.95
MRR: 0.8111666666666667


In [52]:
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=1.0)

all_results_b10 = {}

for q in questions:
    all_results_b10[q["question_id"]] = bm25_search(q["question"], k=10)

for k in [1, 3, 5, 10]:
    print(f"Recall@{k}:", recall_at_k(questions, all_results_b10, k))

print("MRR:", mrr(questions, all_results_b10))

Recall@1: 0.78
Recall@3: 0.87
Recall@5: 0.93
Recall@10: 0.96
MRR: 0.8405119047619047


In [53]:
misses = [q for q in questions if q["gold_passage_id"] not in all_results[q["question_id"]][:5]]
print(len(misses), "questions missed at k=5")

for q in misses[:5]:
    print("Q:", q["question"])
    print("Gold answer:", q["gold_answers"])
    print("Gold passage:", corpus[q["gold_passage_id"]]["text"][:200])
    print("-" * 40)

9 questions missed at k=5
Q: Why would a teacher's college exist?
Gold answer: ['to serve and protect the public interest', 'serve and protect the public', 'to serve and protect the public interest']
Gold passage: There are a variety of bodies designed to instill, preserve and update the knowledge and professional standing of teachers. Around the world many governments operate teacher's colleges, which are gene
----------------------------------------
Q: What was Tesla likely to do with his work?
Gold answer: ['seclude himself', 'seclude himself', 'seclude himself with his work']
Gold passage: Tesla was asocial and prone to seclude himself with his work. However, when he did engage in a social life, many people spoke very positively and admiringly of Tesla. Robert Underwood Johnson describe
----------------------------------------
Q: What is th elast name of the player who was the Super Bowl 50 winner's leading rusher?
Gold answer: ['Anderson', 'Anderson', 'Anderson']
Gold passage: Man

In [54]:
with open("results/bm25_top10.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2)

In [57]:
import csv

with open("results/bm25_experiments.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow([
        "run_name",
        "k1",
        "b",
        "stopwords_removed",
        "recall@1",
        "recall@3",
        "recall@5",
        "recall@10",
        "mrr"
    ])

In [58]:
!ls results

bm25_experiments.csv  bm25_top10.json


In [59]:
!git add notebooks/02_bm25.ipynb results/bm25_top10.json results/bm25_experiments.csv

In [60]:
!git commit -m "Add BM25 retrieval and evaluation"

[nehna 98189d7] Add BM25 retrieval and evaluation
 2 files changed, 1203 insertions(+)
 create mode 100644 results/bm25_experiments.csv
 create mode 100644 results/bm25_top10.json


In [61]:
!git push

Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.73 KiB | 3.73 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/nehnamehranmk638-dev/multilingual-rag-research.git
   399d5ab..98189d7  nehna -> nehna


In [72]:
!git status

On branch nehna
Your branch is up to date with 'origin/nehna'.

nothing to commit, working tree clean
